In [ ]:
def mostrar_integrantes():
    print("=" * 70)
    print("Interpolación polinomial por Diferencias de Newton")
    print("-" * 70)
    print("Integrantes del equipo:")
    print("1. Williams Islas Hernández")
    print("2. Luis Alberto Avendaño Zarza")
    print("=" * 70)
    print()


def menu_principal():
    while True:
        mostrar_integrantes()
        print("MENÚ PRINCIPAL")
        print("1. Interpolación polinomial por Diferencias de Newton")
        print("2. Salir")

        opcion = input("Seleccione una opción: ").strip()

        if opcion == "1":
            modulo_interpolacion_newton()
        elif opcion == "2":
            print("\nSaliendo del programa...")
            break
        else:
            print("\nOpción no válida. Intente de nuevo.\n")


# ------------------------------------------------------------
# FUNCIONES DE ENTRADA
# ------------------------------------------------------------

def leer_entero_positivo(mensaje):
    while True:
        try:
            valor = int(input(mensaje))
            if valor <= 0:
                print("Error: debe ingresar un entero positivo.")
            else:
                return valor
        except ValueError:
            print("Error: entrada inválida. Ingrese un número entero.")


def leer_entero_no_negativo(mensaje):
    while True:
        try:
            valor = int(input(mensaje))
            if valor < 0:
                print("Error: debe ingresar un entero mayor o igual a cero.")
            else:
                return valor
        except ValueError:
            print("Error: entrada inválida. Ingrese un número entero.")


def leer_real(mensaje):
    while True:
        try:
            return float(input(mensaje))
        except ValueError:
            print("Error: entrada inválida. Ingrese un número real.")


def respuesta_si_no(mensaje):
    while True:
        r = input(mensaje).strip().upper()
        if r in ("S", "SI", "SÍ"):
            return True
        elif r in ("N", "NO"):
            return False
        else:
            print("Respuesta inválida. Escriba S o N.")


# ------------------------------------------------------------
# UTILIDADES DE TABLA
# ------------------------------------------------------------

def imprimir_tabla_puntos(x, y):
    print("\nTabla de valores:")
    print("{:<8} {:<15} {:<15}".format("Fila", "xi", "yi"))
    print("-" * 42)
    for i in range(len(x)):
        print("{:<8} {:<15.8f} {:<15.8f}".format(i, x[i], y[i]))
    print()


def ordenar_puntos(x, y):
    pares = list(zip(x, y))
    pares.sort(key=lambda par: par[0])
    x_ordenado = [par[0] for par in pares]
    y_ordenado = [par[1] for par in pares]
    return x_ordenado, y_ordenado


def hay_x_repetidos(x, tolerancia=1e-12):
    for i in range(len(x) - 1):
        if abs(x[i + 1] - x[i]) < tolerancia:
            return True
    return False


def es_equidistante(x, tolerancia=1e-9):
    if len(x) < 2:
        return True, 0.0

    h = x[1] - x[0]
    for i in range(1, len(x) - 1):
        if abs((x[i + 1] - x[i]) - h) > tolerancia:
            return False, h
    return True, h


def obtener_errores_tabla(x, y):
    # se juntan los errores de la tabla.
    errores = []

    if x is None or y is None:
        errores.append("La tabla no existe.")
        return errores

    if len(x) != len(y):
        errores.append("Las listas x e y no tienen la misma longitud.")

    if len(x) < 2:
        errores.append("Se requieren al menos 2 puntos.")

    ordenados = all(x[i] <= x[i + 1] for i in range(len(x) - 1))
    if not ordenados:
        errores.append("Los valores de x no están ordenados.")

    if hay_x_repetidos(x):
        errores.append("Hay valores de x repetidos.")

    if len(x) >= 2:
        equidistante, _ = es_equidistante(x)
        if not equidistante:
            errores.append("La tabla no es equidistante.")

    return errores


# ------------------------------------------------------------
# DIFERENCIAS FINITAS
# ------------------------------------------------------------

def construir_tabla_diferencias_finitas(y):
    n = len(y)
    tabla = [[0.0 for _ in range(n)] for _ in range(n)]

    for i in range(n):
        tabla[i][0] = y[i]

    for j in range(1, n):
        for i in range(n - j):
            tabla[i][j] = tabla[i + 1][j - 1] - tabla[i][j - 1]

    return tabla


def imprimir_tabla_diferencias_finitas(x, tabla):
    n = len(x)
    encabezados = ["xi", "f(xi)"] + [f"Δ^{j}f" if j > 1 else "Δf" for j in range(1, n)]
    ancho = 14

    print("\nTabla de diferencias finitas:")
    for enc in encabezados:
        print(f"{enc:<{ancho}}", end="")
    print()
    print("-" * (ancho * len(encabezados)))

    for i in range(n):
        print(f"{x[i]:<{ancho}.8f}", end="")
        for j in range(n - i):
            print(f"{tabla[i][j]:<{ancho}.8f}", end="")
        print()
    print()


# ------------------------------------------------------------
# SELECCIÓN DE PUNTOS "ÓPTIMOS"
# ------------------------------------------------------------

def seleccionar_subtabla(x, y, xp, grado):

    m = grado + 1
    n = len(x)

    if m > n:
        return None, None, None

    indice_cercano = min(range(n), key=lambda i: abs(x[i] - xp))

    inicio = indice_cercano - grado // 2
    fin = inicio + m - 1

    if inicio < 0:
        inicio = 0
        fin = m - 1

    if fin >= n:
        fin = n - 1
        inicio = n - m

    x_sub = x[inicio:fin + 1]
    y_sub = y[inicio:fin + 1]
    return x_sub, y_sub, inicio


# ------------------------------------------------------------
# EVALUACIÓN NEWTON PROGRESIVO Y REGRESIVO
# ------------------------------------------------------------

def factorial(n):
    resultado = 1
    for i in range(2, n + 1):
        resultado *= i
    return resultado


def evaluar_newton_progresivo(x_sub, tabla, xp, grado, h):
    x0 = x_sub[0]
    s = (xp - x0) / h

    resultado = tabla[0][0]
    producto = 1.0


    terminos = []

    terminos.append({
        "orden": 0,
        "dato": tabla[0][0],
        "factor": "1",
        "divisor": 1,
        "valor": tabla[0][0]
    })

    for j in range(1, grado + 1):
        producto *= (s - (j - 1))
        valor = (producto / factorial(j)) * tabla[0][j]
        resultado += valor

        if j == 1:
            factor = "s"
        else:
            factor = "s"
            for k in range(1, j):
                factor += f"(s-{k})"

        terminos.append({
            "orden": j,
            "dato": tabla[0][j],
            "factor": factor,
            "divisor": factorial(j),
            "valor": valor
        })

    return resultado, s, terminos


def evaluar_newton_regresivo(x_sub, tabla, xp, grado, h):
    n = len(x_sub)
    xn = x_sub[-1]
    s = (xp - xn) / h

    resultado = tabla[n - 1][0]
    producto = 1.0

    terminos = []

    terminos.append({
        "orden": 0,
        "dato": tabla[n - 1][0],
        "factor": "1",
        "divisor": 1,
        "valor": tabla[n - 1][0]
    })

    for j in range(1, grado + 1):
        producto *= (s + (j - 1))
        dato = tabla[n - 1 - j][j]
        valor = (producto / factorial(j)) * dato
        resultado += valor

        if j == 1:
            factor = "s"
        else:
            factor = "s"
            for k in range(1, j):
                factor += f"(s+{k})"

        terminos.append({
            "orden": j,
            "dato": dato,
            "factor": factor,
            "divisor": factorial(j),
            "valor": valor
        })

    return resultado, s, terminos


def elegir_metodo(x_sub, xp):
    distancia_inicio = abs(xp - x_sub[0])
    distancia_final = abs(x_sub[-1] - xp)

    if distancia_inicio <= distancia_final:
        return "progresivo"
    return "regresivo"


def polinomio_texto_progresivo(x_sub, tabla, grado, h):
    partes = [f"{tabla[0][0]:.8f}"]
    for j in range(1, grado + 1):
        coef = tabla[0][j] / factorial(j)
        termino = f"{coef:.8f}"
        termino += "(s)"
        for k in range(1, j):
            termino += f"(s-{k})"
        partes.append(termino)
    return " + ".join(partes), x_sub[0], h


def polinomio_texto_regresivo(x_sub, tabla, grado, h):
    n = len(x_sub)
    partes = [f"{tabla[n - 1][0]:.8f}"]
    for j in range(1, grado + 1):
        coef = tabla[n - 1 - j][j] / factorial(j)
        termino = f"{coef:.8f}"
        termino += "(s)"
        for k in range(1, j):
            termino += f"(s+{k})"
        partes.append(termino)
    return " + ".join(partes), x_sub[-1], h


def imprimir_desarrollo_terminos(terminos):
    # resultado paso por paso.
    print("\nDesarrollo por términos:")
    print("{:<10} {:<18} {:<18} {:<12} {:<18}".format("Orden", "Dato", "Factor", "Divisor", "Valor"))
    print("-" * 76)
    for t in terminos:
        print("{:<10} {:<18.8f} {:<18} {:<12} {:<18.8f}".format(
            t["orden"], t["dato"], t["factor"], t["divisor"], t["valor"]
        ))
    print()


# ------------------------------------------------------------
# LECTURA DE TABLA
# ------------------------------------------------------------

def leer_tabla():
    n = leer_entero_positivo("Ingrese el número de puntos en la tabla: ")
    while n < 2:
        print("Se requieren al menos 2 puntos.")
        n = leer_entero_positivo("Ingrese el número de puntos en la tabla: ")

    x = []
    y = []

    print("\nCapture los puntos (xi, yi):")
    for i in range(n):
        xi = leer_real(f"x[{i}] = ")
        yi = leer_real(f"y[{i}] = ")
        x.append(xi)
        y.append(yi)

    while True:
        imprimir_tabla_puntos(x, y)

        if respuesta_si_no("¿Son correctos los datos? (S/N): "):
            break

        fila = leer_entero_no_negativo("Ingrese la fila a corregir: ")
        while fila < 0 or fila >= n:
            print("Error: fila fuera de rango.")
            fila = leer_entero_no_negativo("Ingrese la fila a corregir: ")

        x[fila] = leer_real(f"Nuevo x[{fila}] = ")
        y[fila] = leer_real(f"Nuevo y[{fila}] = ")

    while True:
        ordenados = all(x[i] <= x[i + 1] for i in range(len(x) - 1))
        if not ordenados:
            print("\nLos datos no estaban ordenados. Se ordenarán automáticamente.")
            x, y = ordenar_puntos(x, y)

        imprimir_tabla_puntos(x, y)


        errores = obtener_errores_tabla(x, y)

        if not errores:
            equidistante, h = es_equidistante(x)
            print(f"Los datos están ordenados y son equidistantes con h = {h:.8f}\n")
            return x, y

        print("La tabla tiene los siguientes problemas:")
        for error in errores:
            print("-", error)
        print()

        if not respuesta_si_no("¿Desea corregir una fila? (S/N): "):
            return None, None

        fila = leer_entero_no_negativo("Ingrese la fila que desea corregir: ")
        while fila < 0 or fila >= n:
            print("Error: fila fuera de rango.")
            fila = leer_entero_no_negativo("Ingrese la fila que desea corregir: ")

        x[fila] = leer_real(f"Nuevo x[{fila}] = ")
        y[fila] = leer_real(f"Nuevo y[{fila}] = ")


# ------------------------------------------------------------
# INTERPOLACIÓN
# ------------------------------------------------------------

def interpolar_con_tabla(x, y):
    while True:
        print("\n--- NUEVA INTERPOLACIÓN CON LA MISMA TABLA ---")
        print(f"Intervalo válido: [{x[0]:.8f}, {x[-1]:.8f}]")

        xp = leer_real("Ingrese el punto a interpolar: ")
        while xp < x[0] or xp > x[-1]:
            print("Error: el punto está fuera del intervalo [x0, xn].")
            xp = leer_real("Ingrese un punto dentro del intervalo: ")

        grado = leer_entero_no_negativo("Ingrese el grado del polinomio: ")

        # revisa si el grado
        while grado < 1 or grado >= len(x):
            if grado < 1:
                print("Error: el grado debe ser mayor o igual a 1.")
            else:
                print(f"Error: con {len(x)} puntos, el grado máximo permitido es {len(x)-1}.")
            grado = leer_entero_no_negativo("Ingrese otro grado: ")

        x_sub, y_sub, inicio = seleccionar_subtabla(x, y, xp, grado)

        print("\nPuntos seleccionados para la interpolación:")
        imprimir_tabla_puntos(x_sub, y_sub)

        h = x_sub[1] - x_sub[0] if len(x_sub) > 1 else 0.0
        tabla = construir_tabla_diferencias_finitas(y_sub)
        imprimir_tabla_diferencias_finitas(x_sub, tabla)

        metodo = elegir_metodo(x_sub, xp)

        if metodo == "progresivo":
            resultado, s, terminos = evaluar_newton_progresivo(x_sub, tabla, xp, grado, h)
            poli, base, h_texto = polinomio_texto_progresivo(x_sub, tabla, grado, h)

            print("Método seleccionado: Newton progresivo")
            print(f"s = (x - x0)/h = (x - {base:.8f})/{h_texto:.8f}")
            print("Polinomio en función de s:")
            print(f"P(x) = {poli}")

        else:
            resultado, s, terminos = evaluar_newton_regresivo(x_sub, tabla, xp, grado, h)
            poli, base, h_texto = polinomio_texto_regresivo(x_sub, tabla, grado, h)

            print("Método seleccionado: Newton regresivo")
            print(f"s = (x - xn)/h = (x - {base:.8f})/{h_texto:.8f}")
            print("Polinomio en función de s:")
            print(f"P(x) = {poli}")

        print(f"\nValor de s = {s:.8f}")
        imprimir_desarrollo_terminos(terminos)
        print("Resultado de la interpolación:")
        print(f"f({xp:.8f}) ≈ {resultado:.10f}\n")

        # ligero cambio al menu
        print("¿Qué desea hacer ahora?")
        print("1. Interpolar otro punto con la misma tabla")
        print("2. Cambiar la tabla")
        print("3. Regresar al menú principal")

        opcion = input("Seleccione una opción: ").strip()
        while opcion not in ("1", "2", "3"):
            print("Opción no válida.")
            opcion = input("Seleccione una opción: ").strip()

        if opcion == "1":
            continue
        elif opcion == "2":
            return "cambiar"
        else:
            return "menu"


# ------------------------------------------------------------
# MÓDULO PRINCIPAL
# ------------------------------------------------------------

def modulo_interpolacion_newton():
    while True:
        print("\n" + "=" * 70)
        print("INTERPOLACIÓN POLINOMIAL POR DIFERENCIAS DE NEWTON")
        print("=" * 70)

        x, y = leer_tabla()

        if x is None or y is None:
            print()
            return

        # seguir, cambiar tabla o salir.
        accion = interpolar_con_tabla(x, y)

        if accion == "cambiar":
            continue
        else:
            print()
            return


if __name__ == "__main__":
    menu_principal()